# Company Dimension Loader

This notebook maintains the `warehouse.dim_company` dimension table.

**Purpose**: Track canonical company entities with stable keys

**Key Features**:
* Stable surrogate keys for companies
* Canonical company name resolution via semantic layer
* Company metadata and attributes
* SCD Type 1 (overwrite on change)

**Source**: `semantic.sem_company_canonical`

**Target**: `workspace.warehouse.dim_company`

In [0]:
%sql
-- Extract canonical company data from semantic layer
CREATE OR REPLACE TEMP VIEW company_extract AS
SELECT DISTINCT
  canonical_company_name,
  company_match_method,
  MAX(company_match_confidence) as max_confidence,
  COUNT(DISTINCT company_name_norm) as alias_count,
  MAX(active_flag) as is_active
FROM workspace.semantic.sem_company_canonical
WHERE canonical_company_name IS NOT NULL
GROUP BY canonical_company_name, company_match_method;

In [0]:
%sql
-- Enrich companies with sector information from semantic company map
CREATE OR REPLACE TEMP VIEW company_enriched AS
SELECT 
  c.*,
  m.sector_name,
  s.sector_sk
FROM company_extract c
LEFT JOIN workspace.semantic.sem_company_map m
  ON c.canonical_company_name = m.canonical_company_name
LEFT JOIN workspace.warehouse.dim_sector s
  ON m.sector_name = s.sector_name;

In [0]:
%sql
-- Generate or maintain stable surrogate keys
CREATE OR REPLACE TEMP VIEW company_with_keys AS
SELECT
  COALESCE(d.company_sk, ROW_NUMBER() OVER (ORDER BY c.canonical_company_name) + COALESCE(max_sk, 0)) as company_sk,
  c.canonical_company_name as company_name,
  c.canonical_company_name as company_name_canonical,
  c.company_match_method,
  c.max_confidence as match_confidence,
  c.alias_count,
  COALESCE(c.sector_sk, -1) as sector_sk,
  c.sector_name,
  COALESCE(c.is_active, TRUE) as is_active,
  CURRENT_TIMESTAMP() as updated_at
FROM company_enriched c
LEFT JOIN workspace.warehouse.dim_company d
  ON c.canonical_company_name = d.company_name_canonical
CROSS JOIN (
  SELECT COALESCE(MAX(company_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_company
) max_keys;

In [0]:
%sql
-- Merge into target dimension (SCD Type 1)
MERGE INTO workspace.warehouse.dim_company target
USING company_with_keys source
ON target.company_name_canonical = source.company_name_canonical
WHEN MATCHED THEN UPDATE SET
  target.company_name = source.company_name,
  target.company_match_method = source.company_match_method,
  target.match_confidence = source.match_confidence,
  target.alias_count = source.alias_count,
  target.sector_sk = source.sector_sk,
  target.sector_name = source.sector_name,
  target.is_active = source.is_active,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  company_sk,
  company_name,
  company_name_canonical,
  company_match_method,
  match_confidence,
  alias_count,
  sector_sk,
  sector_name,
  is_active,
  updated_at
) VALUES (
  source.company_sk,
  source.company_name,
  source.company_name_canonical,
  source.company_match_method,
  source.match_confidence,
  source.alias_count,
  source.sector_sk,
  source.sector_name,
  source.is_active,
  source.updated_at
);

In [0]:
%sql
-- Validate company dimension
SELECT 
  COUNT(*) as total_companies,
  COUNT(DISTINCT sector_sk) as sectors_represented,
  AVG(match_confidence) as avg_match_confidence,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_companies
FROM workspace.warehouse.dim_company;

-- Sample companies
SELECT 
  company_sk,
  company_name,
  sector_name,
  match_confidence,
  alias_count,
  is_active
FROM workspace.warehouse.dim_company
ORDER BY company_sk
LIMIT 20;